---
title: Week 3.7 Tutorial 2, Orchestra Simulation
subtitle: Orchestra Scenario, Carbonate, Water and Calcite simulation
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-07
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Orchestra as a tool
This example illustrates how to use Python and ORCHESTRA to simulate the Carbonate-Water-Calcite system. This system is discussed extensively in Appelo and Postma chapter 5.
The next python cells imports the necessary libararies and checks the path to the input files required for the Orchestra simulation. Please note that this is only required for the jupyter-book version. For a stand-alone version of this notebook you need to start jupyter-lab from the directory with the input files.

In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

# Prepare a file to capture PyOrchestra output
capture_file = open("pyorchestra_output.log", "w")


# pyOrchestra is implemented in C++
# Save original stdout file descriptor
original_stdout_fd = sys.stdout.fileno()

# Duplicate original stdout so we can restore it later
saved_stdout_fd = os.dup(original_stdout_fd)



# We need to import some Orchestra files. We need to know the path layout on 
# the local machine:
def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:


# Example usage:
# input_file = path_from_book_root("content", "week 02", "Orchestra_simulation", "chemistry1.inp")
# print("Input file:", input_file)
   
# orchestra_path = path_from_book_root("content", "week 3.7", "Week 3.7_Tutorial-2-Orchestra-Carbonate-system")
orchestra_path = "/home/theimovaara/data_ssd/GitLab/oit_chemistry_for_earth_sciences/chemistry_book/content/project_THe/Orchestra_Carbonate_Test"
# print(orchestra_path)

## Download this script and the required ORCHESTRA files
TODO Create an yml for a local python environment
TODO Create zip file with the correct files. Explain how to install Orchestra using pip in a local python environment.
TODO Update all files so that they are well documented and describe the system

pyorchestra will be initialize with the correct chemistry file...

explain that InVars are the species used to define the totals. 
explain that OutVars is the list of Orchestra variables that are exported to python for post-processing or necessary for simulation.

Once prinicple problem is known we can create and initialize a pyorchestra object.

## Define chemical system: Carbonate, Water and Calcite using the ORCHESTRA-GUI
In order to solve a chemical equilibrium problem with pyOrchestra, we need to define our chemical system first. Orchestra defines this system with a number of text files. The most important one is the so-called chemistry input file (*chemistry1.inp*). This file is most easily made using the Orchestra GUI which can be accessed by clicking on the orchestra2023.jar file.

For the carbonate system we distinguish three types of scenarios: 
1. a scenario with a gas-phase with a fixed closed volume. This implies that the pressure in the gas phase will change based on the chemical processes occuring in the system;
2. a scenario where the gas pressure is assumed to be constant in a closed volume, this implies that the volume of the system changes based on the chemical processes;
3. a scenario where our system is in open contact with the atmosphere, this implies that the partial pressures (and therefore the activities) of the gaseous components do not change as the reactions proceed in our system. This because the volume (and therefore the amount of gaseous species present) is so much larger than our system that we may neglect the changes due to our reactions. For an open reaction in equilibrium with the atmosphere we need another input file. This case is not required for this tutorial.

Similar to the first tutorial of this week we start with a skeleton problem (a working ORCHESTRA simulation and it is convenient to start with the results from tutorial 1) and then adjust the files step by step. 

Using the steps described in Tutorial 1 we created *chemistry_fixedVolume.inp*. The carbonate system we want to analyse is controlled by the following reactions (see also Appelo & Postma, chapter 5):
$$
\begin{aligned}
\text{CO}_2\text{(g)} + \text{H}_2\text{O} &\leftrightharpoons \text{H}_2\text{CO}_3^* \\
\text{H}_2\text{CO}_3^* &\leftrightharpoons \text{H}^+ + \text{HCO}_3^- \\
\text{HCO}_3^- &\leftrightharpoons \text{H}^+ + \text{CO}_3^{-2} \\
\text{H}_2\text{O} &\leftrightharpoons \text{H}^+ + \text{OH}^- \\
\text{CaCO}_3\text{(s)} &\leftrightharpoons \text{Ca}^{+2} + \text{CO}_3^{-2} \\
\end {aligned}
$$

As always when analysing chemical equilibria, all of the reactions are taking place simultaneously, and if we were to add all the reactions together we would obtain the following net reaction:
$$
\begin{aligned}
\text{CO}_2\text{(g)} + \text{H}_2\text{O} + \text{CaCO}_3\text{(s)} &\leftrightharpoons \text{Ca}^{+2} + 2\text{H}\text{CO}_3^- \\
\end {aligned}
$$


```{exercise} Check this calculation
See if you can obtain the above net reaction yourself?
```
From the above net reaction we observe that the Carbonate in the system orginates from $\text{CO}_2\text{g}$ and $\text{CaCO}_3\text{(s)}$, while $\text{Ca}^{+2}$ only originates from $\text{CaCO}_3\text{(s)}$.

As we will be working with a system that contains three phases (gas, water, and solids), we require a back ground gas to maintain gas pressure in the case all of the gaseous species in our reaction were to be consumed. For our case we choose to use Ar[g] at a background pressure of 1 atm. Argon is an inert gas and will not influence our reactions.


```{exercise} Why is it important to have a background gas?
```
Finally we require Orchestra to control the charge balance, we assume that $\text{Na}^+$ and $\text{Cl}^-$ are present in the sytem as well because we assume that we will be able to change the pH in the solution using $\text{HCl}$ or $\text{NaOH}$.

Using all insights from above we generated the following master species table: 

|Primary entity|Phase|Input Variable|Fix log activity|Log activity|Concentration|Phase|Expression|
|--|--|--|--|--|--|--|--|
|Ar[g]|gas||x|0||||
|CO3-2|diss||||1.0e-9|tot||
|Ca+2|diss||||1.0e-9|tot||
|Cl-|diss||x||1.0e-9|tot||
|H+|diss|pH||7.0|||H.logact=-pH|
|H2O|liter||x|-0.0||||
|Na+|diss||x||1.0e-9|tot||

Orchestra will use these primary entities, and their initial values to calculate the total elemental composition in the complete system. The fixed log-activties indicate that the total can vary during the simulation, as long as the log-activity condition is maintained or because Orchestra uses these values to balance the charge of the solution (Na+.tot and Cl-.tot). 

### Phases & Reactions
On the **Phases & Reactions** tab shown in figure [](#phases_reactions) we define the reaction network of our system. Please ensure that all reactions are marked.

### Other tabs
There is no need to change anything on the other tabs as these are correctly set in Tutorial 1.
There is however one thing that you need to check before we can proceed which is the way ORCHESTRA calculates the pressure in the gas phase. This cannot be done with the GUI, but has to done by adding (or unmarking) code in the *chemistry.inp* file itself. This is the reason why we now use *chemistry_fixedVolume.inp* instead of *chemistry1.inp* because we have changed line 68 to
```{code}
@Calc: (5,  "pressure = {CO2[g].con} + {Ar[g].con}")     // is total fluid volume in cell
```
allowing ORCHESTRA to calculate the total pressure as the sum of the partial pressures of the gaseous species defined by us.

For a fixed-pressure problem we need to change this file in a different way as is done in *chemistry_fixedPressure.inp*.

### Using Orchestra to run the scenario
It is possible to use the ORCHESTRA GUI to run the simulation, and this has been prepared using the **Input** and **Output** tabs on the right of the window. Please have look at these tabs and they should be rather "self explanatory". 



## Running the Carbonate-Water-Calcite scenario in a Python notebook
We will use a Python notebook and ORCHESTRA to replicate the graphs shown in Appelo and Postma chapter 5. In addition we will run some test scenarios to illustrate the importance of carbonate and $\text{CO}_2\text{(g)}$ in controlling the pH of water.

We follow the same systematic approach as Tutorial 1, which consists of the following standard steps:
1. define the domain for our problem
2. define the primary species which change during our scenario
3. initialize the problem
4. run the problem
5. process the output

### 1. Define the domain
For our analysis we choose to work with a system containing 1 liter of water and 1 liter of gas phase. The volume of the solid phase is small with respect the total volume and will be neglected for now.

### 2. Define the primary species
The primary species are as defined above with the ORCHESTRA GUI. Please note that the values required by ORCHESTRA have to be in moles. In order to get moles/liter the volume of water has to be set to 1 liter, and the density of water need to be given as well.

### 3. Initialise the problem 
The initial condition for our problem will be defined by setting the amounts of $\text{CO}_2\text{(g)}$ and $\text{CaCO}_3\text{(s)}$ in our system with using CO2[g].tot and CaCO3[s].tot. 

Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry1.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code shows how to do this.
```{note} 
It is relatively easy to change the OutVars array. In order to find all possible variables in ORCHESTRA, check the **Phases & Reactions** tab on the **Chemistry** tab in the ORCHESTRA-GUI .
```


In [2]:
invars = [
    'Ar[g].logact','HCO3-.tot','Ca+2.tot','Cl-.tot',
    'Na+.tot','pH','watervolume','gasvolume_fixed','gas_type',
    'fixed_logact_CO2'
    ]

invars_min = [
    'Ar[g].min','HCO3-.min','Ca+2.min','Cl-.min',
    'Na+.min',
]

invars_gas = [
    'Ar[g].gas','HCO3-.gas','Ca+2.gas','Cl-.gas',
    'Na+.gas',
]

invars_diss = [
    'Ar[g].diss','HCO3-.diss','Ca+2.diss','Cl-.diss',
    'Na+.diss',
]


outvars_tot = [
    'Alkalinity.tot','Ar.tot','Ar[g].tot','C.tot','CO2.tot',
    'CO2[g].tot','CO2g[s].tot','CO3-2.tot','C[+4].tot','Ca.tot',
    'Ca+2.tot','CaCO3.tot','CaHCO3+.tot','CaOH+.tot','Ca[OH]+.tot',
    'Calcite[s].tot','Cl.tot','Cl-.tot','H.tot','H+.tot','H2CO3.tot',
    'H2O.tot','H2O[g].tot','HCO3-.tot','H[+1].tot','Na.tot',
    'Na+.tot','NaCO3-.tot','NaHCO3.tot','O.tot','OH-.tot',
    'O[-2].tot','[CO2]2.tot',
]

outvars_logact = [
    'Alkalinity.logact','Ar.logact','Ar[g].logact','C.logact','CO2.logact',
    'CO2[g].logact','CO2g[s].logact','CO3-2.logact','C[+4].logact','Ca.logact',
    'Ca+2.logact','CaCO3.logact','CaHCO3+.logact','CaOH+.logact','Ca[OH]+.logact',
    'Calcite[s].logact','Cl.logact','Cl-.logact','H.logact','H+.logact','H2CO3.logact',
    'H2O.logact','H2O[g].logact','HCO3-.logact','H[+1].logact','Na.logact',
    'Na+.logact','NaCO3-.logact','NaHCO3.logact','O.logact','OH-.logact',
    'O[-2].logact','[CO2]2.logact',
]

outvars_con = [
    'Alkalinity.con','Ar.con','Ar[g].con','C.con','CO2.con',
    'CO2[g].con','CO2g[s].con','CO3-2.con','C[+4].con','Ca.con',
    'Ca+2.con','CaCO3.con','CaHCO3+.con','CaOH+.con','Ca[OH]+.con',
    'Calcite[s].con','Cl.con','Cl-.con','H.con','H+.con','H2CO3.con',
    'H2O.con','H2O[g].con','HCO3-.con','H[+1].con','Na.con',
    'Na+.con','NaCO3-.con','NaHCO3.con','O.con','OH-.con',
    'O[-2].con','[CO2]2.con',
]

outvars_tot_con = [
    'Alkalinity.tot.con','Ar.tot.con','Ar[g].tot.con','C.tot.con','CO2.tot.con',
    'CO2[g].tot.con','CO2g[s].tot.con','CO3-2.tot.con','C[+4].tot.con','Ca.tot.con',
    'Ca+2.tot.con','CaCO3.tot.con','CaHCO3+.tot.con','CaOH+.tot.con','Ca[OH]+.tot.con',
    'Calcite[s].tot.con','Cl.tot.con','Cl-.tot.con','H.tot.con','H+.tot.con','H2CO3.tot.con',
    'H2O.tot.con','H2O[g].tot.con','HCO3-.tot.con','H[+1].tot.con','Na.tot.con',
    'Na+.tot.con','NaCO3-.tot.con','NaHCO3.tot.con','O.tot.con','OH-.tot.con',
    'O[-2].tot.con','[CO2]2.tot.con',
]


outvars_min_si = [
    'Aragonite[s].si','CO2g[s].si','Calcite[s].si','Halite[s].si',
]



In [3]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_new.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    InVars1 = np.array(invars)
    
    # We select the output from Orchestra we need to use
    # pleaste note that lists of strings can be concatenated using the + operator.
    out_list = (
        invars + outvars_logact + outvars_tot + outvars_con +
        outvars_min_si + outvars_tot_con + 
        invars_diss + invars_gas + invars_min +
        ['I', 'chargebalance', 'totcharge', 'gasvolume', 'pressure'])
        
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)



Reading and expanding calculator new stylechemistry_new.inp
Scanning file: chemistry_new.inp
Scanning file: objects2025_THe.txt
Scanning file: chemistry_new.inp
Scanning file: objects2025_THe.txt
Including file: chemistry_new.inp
Scanning file: objects2025_THe.txt
0.02 sec.
	Reading variables .... 0.008 s
testing:
8:Ar[g].logact
15:HCO3-.tot
10:Ca+2.tot
12:Cl-.tot
17:Na+.tot
13:pH
20:watervolume
21:gasvolume_fixed
22:gas_type
23:fixed_logact_CO2
8:Ar[g].logact
15:HCO3-.tot
10:Ca+2.tot
12:Cl-.tot
17:Na+.tot
13:pH
20:watervolume
21:gasvolume_fixed
22:gas_type
23:fixed_logact_CO2
24:Alkalinity.logact
25:Ar.logact
8:Ar[g].logact
26:C.logact
27:CO2.logact
28:CO2[g].logact
29:CO2g[s].logact
30:CO3-2.logact
31:C[+4].logact
32:Ca.logact
9:Ca+2.logact
33:CaCO3.logact
34:CaHCO3+.logact
35:CaOH+.logact
36:Ca[OH]+.logact
37:Calcite[s].logact
38:Cl.logact
11:Cl-.logact
39:H.logact
40:H+.logact
41:H2CO3.logact
2:H2O.logact
42:H2O[g].logact
14:HCO3-.logact
43:H[+1].logact
44:Na.logact
16:Na+.logact
4

In [4]:
OutVars1

array(['Ar[g].logact', 'HCO3-.tot', 'Ca+2.tot', 'Cl-.tot', 'Na+.tot',
       'pH', 'watervolume', 'gasvolume_fixed', 'gas_type',
       'fixed_logact_CO2', 'Alkalinity.logact', 'Ar.logact',
       'Ar[g].logact', 'C.logact', 'CO2.logact', 'CO2[g].logact',
       'CO2g[s].logact', 'CO3-2.logact', 'C[+4].logact', 'Ca.logact',
       'Ca+2.logact', 'CaCO3.logact', 'CaHCO3+.logact', 'CaOH+.logact',
       'Ca[OH]+.logact', 'Calcite[s].logact', 'Cl.logact', 'Cl-.logact',
       'H.logact', 'H+.logact', 'H2CO3.logact', 'H2O.logact',
       'H2O[g].logact', 'HCO3-.logact', 'H[+1].logact', 'Na.logact',
       'Na+.logact', 'NaCO3-.logact', 'NaHCO3.logact', 'O.logact',
       'OH-.logact', 'O[-2].logact', '[CO2]2.logact', 'Alkalinity.tot',
       'Ar.tot', 'Ar[g].tot', 'C.tot', 'CO2.tot', 'CO2[g].tot',
       'CO2g[s].tot', 'CO3-2.tot', 'C[+4].tot', 'Ca.tot', 'Ca+2.tot',
       'CaCO3.tot', 'CaHCO3+.tot', 'CaOH+.tot', 'Ca[OH]+.tot',
       'Calcite[s].tot', 'Cl.tot', 'Cl-.tot', 'H.tot', 'H+.tot

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Run the Problem
The aim of the scenario for now is to see if we can replicate a similar graph as figure 4.1 of Appelo and Postma.

:::{figure} images/Appelo&Postma_fig5.5.png
:name: appelo_fig5_5
:align: center
:width: 75%
Appelo & Postma figure 5.5
:::

In our simulation we control the total amount of carbonate in the system with CO2[g].tot, if we require carbonate in a mineral phase we can add this by defining the total amount of Ca+2.tot in the system, the system will allow Calcite (and other Calcium carbonate minerals to precipitate). As the pH in the figure ranges from 3 to 13 we choose to have pH as the "driving" variable for our scenario.
Finally the total amount of Carbonate in the system is not really relevant as the graph shows a percentage of $\text{HCO}_3^-$, we choose to set CO2[g].tot to 1 mmol in this system.

### Step 1: Define the default InVars

In [5]:
# %%
# Run the initialize pyOrchestra class for the data in df_work

# Initialise the IN1 array and the output matrix (all_Res)
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
all_Res = np.zeros([1,len(OutVars1)])

IN1[0][np.where(InVars1 == 'pH')] = 7 # 
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = 0 # moles
IN1[0][np.where(InVars1 == 'CO3-2.tot')] = 2 # moles
IN1[0][np.where(InVars1 == 'Ca+2.tot')] = 1 # 
IN1[0][np.where(InVars1 == 'Na+.tot')] = 0.01 # 
IN1[0][np.where(InVars1 == 'Cl-.tot')] = 0.01 # 

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = 0 # fixed CO2 logact
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1 # no gas volume present

# run ORCHESTRA
OUT = pO1.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation = pd.DataFrame([all_Res],columns=OutVars1, index=['initial calculation'])


# Display some of the output in a nicely formatted way
# A first analysis will be on the chargebalance, to do this we require the charge balance
# and the total charge.
table_mdini = Res_Simulation[[
    'pH','pressure','Ca+2.diss', 'Na+.diss', 
    'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))





|                     |      pH |   pressure |   Ca+2.diss |   Na+.diss |   CO2[g].logact |   HCO3-.tot |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   HCO3-.con |   CO3-2.con |
|:--------------------|--------:|-----------:|------------:|-----------:|----------------:|------------:|------------:|---------------:|--------------:|----------------:|------------:|------------:|
| initial calculation | 9.98533 |    1.03119 | 0.000151088 |       0.01 |        -6.28019 |           1 |           1 |       -4.11486 |      -4.02149 |     8.88178e-16 | 8.49825e-05 | 5.30304e-05 |

In [6]:
print(InVars1)
print(IN1[0])

['Ar[g].logact' 'HCO3-.tot' 'Ca+2.tot' 'Cl-.tot' 'Na+.tot' 'pH'
 'watervolume' 'gasvolume_fixed' 'gas_type' 'fixed_logact_CO2']
[0.   1.   1.   0.01 0.01 7.   1.   1.   0.   0.  ]


Conclusion, master species name diss is total master-species in dissolved phase

In [7]:
# Check totals different phases
sel_out = ['HCO3-.diss','HCO3-.min','HCO3-.gas']
print(Res_Simulation[sel_out])
print(Res_Simulation[sel_out].sum(axis=1))
print(Res_Simulation['HCO3-.tot'])


                     HCO3-.diss  HCO3-.min     HCO3-.gas
initial calculation    0.000151   0.999849  5.245725e-07
initial calculation    1.0
dtype: float32
                     HCO3-.tot  HCO3-.tot
initial calculation        1.0        1.0


In [8]:
# Check totals different phases
sel_out = ['Ca+2.diss','Ca+2.min','Ca+2.gas']
print(Res_Simulation[sel_out])
print(Res_Simulation[sel_out].sum(axis=1))
print(Res_Simulation['Ca+2.tot'])


                     Ca+2.diss  Ca+2.min  Ca+2.gas
initial calculation   0.000151  0.999849       0.0
initial calculation    1.0
dtype: float32
                     Ca+2.tot  Ca+2.tot
initial calculation       1.0       1.0


SyntaxError: invalid syntax (1987144486.py, line 1)